<a href="https://colab.research.google.com/github/cto-school/mathematics-for-machine-learning/blob/main/08-probability-and-statistics/08b_statistics_distributions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 08b — Statistics & Distributions

In 08a we estimated probabilities by simulation. Now we meet the **named
distributions** that describe the most common kinds of randomness, learn the
language of **PMF / PDF / CDF**, summarise data with **mean, median, std**,
measure how two quantities move together with **covariance and correlation**,
and finish with the spectacular **Central Limit Theorem**.

As before we use one reproducible random generator.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)   # reproducible randomness

## 1. Discrete vs continuous; PMF, PDF, CDF

A random variable is **discrete** if it takes separate values (die faces, counts)
and **continuous** if it can take any value in a range (heights, errors).

- **PMF** (probability *mass* function), for discrete $X$: $p(k) = P(X = k)$.
- **PDF** (probability *density* function), for continuous $X$: a curve $f(x)$
  whose **area** over an interval gives the probability of landing there.
- **CDF** (cumulative distribution function), for *both*:
  $$F(x) = P(X \le x).$$

Key idea: for a continuous variable, $P(X = x) = 0$; only **intervals** have
positive probability, and that probability is the **area under the PDF**.

## 2. The uniform distribution

The **continuous uniform** on $[0, 1]$ is "equally likely anywhere": its PDF is
flat, $f(x) = 1$ for $x \in [0,1]$. `rng.random` samples from it.

In [ ]:
U = rng.random(10_000)        # 10,000 samples from Uniform(0, 1)

plt.figure(figsize=(7, 4))
# density=True scales the histogram so its total area is 1 -> comparable to a PDF
plt.hist(U, bins=20, density=True, edgecolor="black", alpha=0.7,
         label="histogram of samples")
plt.axhline(1.0, color="red", linewidth=2, label="true PDF  f(x) = 1")
plt.title("Uniform(0, 1): flat density")
plt.xlabel("x"); plt.ylabel("density"); plt.legend(); plt.grid(True)
plt.show()

## 3. Bernoulli and binomial

A **Bernoulli($p$)** variable is a single yes/no trial: $1$ with probability $p$,
$0$ otherwise (a biased coin). Add up $n$ independent Bernoulli trials and you
get a **Binomial($n, p$)** variable — the number of successes. Its PMF is

$$P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}.$$

In [ ]:
p = 0.3
# Bernoulli: True/False with prob p, converted to 1/0 with astype(int)
bernoulli = (rng.random(15) < p).astype(int)
print("Bernoulli(0.3) samples:", bernoulli)

# Binomial: count successes in n=10 trials, repeated many times
n = 10
samples = rng.binomial(n, p, size=10_000)   # each value is in 0..10
print("a few Binomial(10, 0.3) samples:", samples[:15])

Let's overlay the simulated frequencies with the exact binomial PMF.

In [ ]:
from math import comb     # comb(n, k) = n-choose-k

n, p = 10, 0.3
samples = rng.binomial(n, p, size=50_000)

ks = np.arange(0, n + 1)
# Exact PMF from the formula, for each k = 0..n
pmf = np.array([comb(n, k) * p**k * (1 - p)**(n - k) for k in ks])
# Estimated PMF: fraction of samples equal to each k
est = np.array([np.mean(samples == k) for k in ks])

plt.figure(figsize=(7, 4))
plt.bar(ks - 0.15, est, width=0.3, label="simulated", color="steelblue")
plt.bar(ks + 0.15, pmf, width=0.3, label="exact PMF", color="orange")
plt.title("Binomial(10, 0.3): simulation vs theory")
plt.xlabel("number of successes k"); plt.ylabel("P(X = k)")
plt.legend(); plt.grid(True, axis="y")
plt.show()

## 4. The normal (Gaussian) distribution

The **normal** distribution $\mathcal{N}(\mu, \sigma^2)$ is the famous bell
curve, with mean $\mu$ and standard deviation $\sigma$. Its PDF is

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}}\, \exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right).$$

We sample with `rng.normal` and overlay the histogram on the exact density.

In [ ]:
mu, sigma = 0.0, 1.0
data = rng.normal(mu, sigma, size=20_000)     # standard normal samples

# Grid of x values for the theoretical curve
x = np.linspace(-4, 4, 400)
pdf = (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-(x - mu)**2 / (2 * sigma**2))

plt.figure(figsize=(7, 4))
plt.hist(data, bins=40, density=True, alpha=0.6, edgecolor="black",
         label="histogram of samples")
plt.plot(x, pdf, "r-", linewidth=2, label="exact PDF")
plt.title("Standard normal N(0, 1)")
plt.xlabel("x"); plt.ylabel("density"); plt.legend(); plt.grid(True)
plt.show()

### The CDF

The CDF $F(x) = P(X \le x)$ climbs from $0$ to $1$. We can estimate it from
samples by sorting them: the fraction of data $\le x$ is the **empirical CDF**.

In [ ]:
data = rng.normal(0, 1, size=2000)
xs = np.sort(data)                       # sorted samples
emp_cdf = np.arange(1, len(xs) + 1) / len(xs)   # 1/N, 2/N, ..., 1

plt.figure(figsize=(7, 4))
plt.plot(xs, emp_cdf, label="empirical CDF")
plt.title("CDF of N(0,1): probability of landing at or below x")
plt.xlabel("x"); plt.ylabel("F(x) = P(X <= x)"); plt.legend(); plt.grid(True)
plt.show()

## 5. Summary statistics: mean, median, std

NumPy gives us the standard summaries directly:

- `np.mean`  — the average $\bar{x} = \frac1N\sum_i x_i$;
- `np.median` — the middle value (robust to outliers);
- `np.std`   — the standard deviation, the typical distance from the mean.

In [ ]:
data = rng.normal(10, 2, size=10_000)    # mean ~10, std ~2

print("mean   :", np.mean(data))
print("median :", np.median(data))
print("std    :", np.std(data))
print("min/max:", np.min(data), np.max(data))

---
## ✍️ Exercise 1 (solution included)

Draw $50{,}000$ samples from a normal with mean $\mu = 5$ and standard deviation
$\sigma = 3$. Report the **sample** mean and std, and check they are close to
$5$ and $3$. Then plot a density histogram with the true PDF overlaid.

**Solution:**

In [ ]:
mu, sigma = 5.0, 3.0
data = rng.normal(mu, sigma, size=50_000)
print("sample mean:", np.mean(data), " sample std:", np.std(data))

x = np.linspace(mu - 4*sigma, mu + 4*sigma, 400)
pdf = (1 / (sigma * np.sqrt(2*np.pi))) * np.exp(-(x - mu)**2 / (2*sigma**2))

plt.figure(figsize=(7, 4))
plt.hist(data, bins=40, density=True, alpha=0.6, edgecolor="black",
         label="samples")
plt.plot(x, pdf, "r-", linewidth=2, label="true PDF")
plt.title("N(5, 9): samples vs density"); plt.xlabel("x"); plt.ylabel("density")
plt.legend(); plt.grid(True)
plt.show()

## 6. Covariance and correlation

For *two* variables we ask: do they move **together**? The **covariance** is

$$\operatorname{Cov}(X, Y) = \mathbb{E}\big[(X-\mu_X)(Y-\mu_Y)\big],$$

positive when they rise together, negative when one rises as the other falls.
Its scale-free version is the **correlation coefficient**
$\rho = \operatorname{Cov}(X,Y) / (\sigma_X \sigma_Y) \in [-1, 1]$.

We'll build $Y$ as "$X$ plus noise" so the two are genuinely related.

In [ ]:
N = 2000
X = rng.normal(0, 1, size=N)
Y = 0.8 * X + rng.normal(0, 0.5, size=N)   # Y depends on X, plus noise

# np.cov / np.corrcoef return 2x2 matrices; the off-diagonal is what we want.
cov_matrix  = np.cov(X, Y)
corr_matrix = np.corrcoef(X, Y)
print("covariance  Cov(X, Y) :", cov_matrix[0, 1])
print("correlation rho(X, Y) :", corr_matrix[0, 1])

plt.figure(figsize=(6, 6))
plt.scatter(X, Y, s=8, alpha=0.4)
plt.title(f"Positively correlated data (rho = {corr_matrix[0,1]:.2f})")
plt.xlabel("X"); plt.ylabel("Y"); plt.grid(True)
plt.show()

## ✍️ Exercise 2 (solution included)

Construct two variables that are **negatively** correlated, e.g. $Y = -X + $
noise, and confirm the correlation coefficient is close to $-1$. Make a scatter
plot.

**Solution:**

In [ ]:
N = 2000
X = rng.normal(0, 1, size=N)
Y = -X + rng.normal(0, 0.3, size=N)        # negative relationship + small noise

rho = np.corrcoef(X, Y)[0, 1]
print("correlation rho(X, Y):", rho)       # should be close to -1

plt.figure(figsize=(6, 6))
plt.scatter(X, Y, s=8, alpha=0.4, color="darkred")
plt.title(f"Negatively correlated data (rho = {rho:.2f})")
plt.xlabel("X"); plt.ylabel("Y"); plt.grid(True)
plt.show()

## 7. The Central Limit Theorem

Here is one of the deepest facts in probability. Take **any** distribution with
a finite mean and variance — it need not look normal at all. Draw a sample of
size $n$ and compute its **mean**. Repeat many times. The **Central Limit
Theorem** (CLT) says: the distribution of these sample means becomes
approximately **normal** as $n$ grows, *regardless of the original shape*.

Let us start from a deliberately non-normal variable: the uniform distribution
on $[0,1]$ (a flat box), and watch its sample means turn into a bell curve.

In [ ]:
# A single uniform variable is flat -- definitely not bell-shaped:
flat = rng.random(20_000)
plt.figure(figsize=(7, 3.5))
plt.hist(flat, bins=30, density=True, edgecolor="black", alpha=0.7)
plt.title("Original variable: Uniform(0,1) -- flat, not normal")
plt.xlabel("value"); plt.ylabel("density"); plt.grid(True)
plt.show()

In [ ]:
# Now take the mean of n=30 uniform samples, 10,000 separate times.
n = 30
num_experiments = 10_000

# Make a (num_experiments x n) array; each ROW is one experiment of n draws.
batches = rng.random((num_experiments, n))
sample_means = batches.mean(axis=1)        # average across each row

# Overlay the normal predicted by the CLT.
# Uniform(0,1) has mean 1/2 and variance 1/12, so the mean of n of them
# has mean 1/2 and std sqrt(1/12 / n).
mu = 0.5
sigma = np.sqrt((1/12) / n)
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 400)
pdf = (1 / (sigma * np.sqrt(2*np.pi))) * np.exp(-(x - mu)**2 / (2*sigma**2))

plt.figure(figsize=(7, 4))
plt.hist(sample_means, bins=40, density=True, alpha=0.6, edgecolor="black",
         label="distribution of sample means")
plt.plot(x, pdf, "r-", linewidth=2, label="normal predicted by the CLT")
plt.title(f"Central Limit Theorem: means of {n} uniforms look normal")
plt.xlabel("sample mean"); plt.ylabel("density"); plt.legend(); plt.grid(True)
plt.show()

The flat box became a bell curve! This is why the normal distribution appears
everywhere in statistics and machine learning: anything that is a **sum or
average of many small independent effects** tends to be approximately normal.

---
## 📝 Homework (no solutions provided)

1. Sample $30{,}000$ values from `rng.binomial(20, 0.5, size=...)`. Plot the
   density histogram and overlay a normal with the same mean and std — the
   binomial is close to normal when $n$ is large.
2. Demonstrate the CLT starting from a **different** non-normal variable, for
   example the exponential distribution `rng.exponential(1.0, size=...)`. Plot
   the histogram of sample means for $n = 50$.
3. Generate $X \sim \mathcal{N}(0,1)$ and a truly **independent**
   $Y \sim \mathcal{N}(0,1)$. Estimate their correlation and explain why it is
   near $0$. Make a scatter plot.
4. For data from `rng.normal(0, 1, size=10000)`, estimate
   $P(-1 \le X \le 1)$ by counting, and compare with the well-known value
   $\approx 0.68$ (the "68% within one standard deviation" rule).